In [38]:
from keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.regularizers import l2
from project_brain_decoder.config import get_project_root
from project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import numpy as np

In [39]:
tf.random.set_seed(42)
np.random.seed(42)

In [40]:
folder = get_project_root() / "data" / "raw"
files = list(folder.glob("*.nwb"))
batch_size, window_size, input_dim = 128, 30, 192

In [41]:
train = files[:187] # 60%
val = files[187:249] # 20%
test = files[249:] # 20%

In [44]:
def make_windows(neural: np.array, # shape(T, C) - time * channels
                 targets: np.array, # shape(T,) or (T, out_dim)
                 window_size: int,
                 stride: int=10) -> tuple[np.array, np.array]:
    """Slice into (window_size, C) windows; targets aligned to last timestep of each window"""
    T, C = neural.shape
    X = np.lib.stride_tricks.sliding_window_view(neural, window_size, axis=0)[::stride] # (n_windows, C, window_size)
    X = X.transpose(0, 2, 1) # (n_windows, window_size, C)
    # target for each window = value at the end of the window
    y = targets[window_size - 1 :: stride][:X.shape[0]]
    return X, y

In [46]:
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()

In [47]:
for file in train:
    session = load_nwb(file_path=file)
    neural = np.concatenate([session["neural_spiking_band"], session["neural_threshold_crossings"]], axis=1)
    targets = np.column_stack([session["target_index_velocity"], session["target_mrs_velocity"]])
    neural_scaler.partial_fit(neural)
    targets_scaler.partial_fit(targets)

In [ ]:
n_train_windows = 0
for file in train:
    session = load_nwb(file)
    neural = session["neural_spiking_band"]
    T = neural.shape[0]
    n_train_windows += T - window_size + 1
n_train_steps = n_train_windows // batch_size
print(n_train_steps)

In [ ]:
n_val_windows = 0
for file in val:
    session = load_nwb(file)
    neural = session["neural_spiking_band"]
    T = neural.shape[0]
    n_val_windows += T - window_size + 1
n_val_steps = n_val_windows // batch_size
print(n_val_steps)

In [ ]:
n_test_windows = 0
for file in test:
    session = load_nwb(file)
    neural = session["neural_spiking_band"]
    T = neural.shape[0]
    n_test_windows += T - window_size + 1
n_test_steps = n_test_windows // batch_size
print(n_test_steps)

In [48]:
def make_dataset(file_list, batch_size, shuffle=False):
    def generator():
        for file in file_list:
            session = load_nwb(file_path=file)
            neural = np.concatenate([session["neural_spiking_band"], session["neural_threshold_crossings"]], axis=1)
            targets = np.column_stack([session["target_index_velocity"], session["target_mrs_velocity"]])
            X_scaled = neural_scaler.transform(neural)
            y_scaled = targets_scaler.transform(targets)
            X_w, y_w = make_windows(X_scaled, y_scaled, window_size)
            for i in range(len(X_w)):
                yield X_w[i], y_w[i]

    ds = tf.data.Dataset.from_generator(generator=generator,
                                        output_signature=(tf.TensorSpec(shape=(window_size, input_dim), dtype=tf.float32), tf.TensorSpec(shape=(2,), dtype=tf.float32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=10_000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train, batch_size, shuffle=True).repeat()
val_ds = make_dataset(val, batch_size).repeat()
test_ds = make_dataset(test, batch_size).repeat()

In [58]:
def get_transformer(window_size, input_dim):
    input_layer = Input(shape=(window_size, input_dim))
    drop_out_1 = Dropout(0.3)(input_layer)
    # Multi head attention
    attention_1 = MultiHeadAttention(num_heads=4, key_dim=48)(drop_out_1, drop_out_1)
    drop_out_2 = Dropout(0.3)(attention_1)
    skip_1 = LayerNormalization()(drop_out_2 + input_layer) # skip connection
    # Feed forward block
    dense_1 = Dense(units=384, activation="relu", kernel_regularizer=l2(0.0001))(skip_1)
    drop_out_3 = Dropout(0.3)(dense_1)
    dense_2 = Dense(units=192, kernel_regularizer=l2(0.0001))(drop_out_3)
    drop_out_4 = Dropout(0.3)(dense_2)
    skip_2 = LayerNormalization()(drop_out_4 + skip_1) # skip connection 2
    avg_pool = GlobalAveragePooling1D()(skip_2)
    drop_out_5 = Dropout(0.3)(avg_pool)
    output = Dense(units=2)(drop_out_5)
    model = Model(inputs=[input_layer], outputs=[output])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss="mse")
    return model

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

In [59]:
def main(model):
    get_transformer(window_size, input_dim).fit(train_ds, epochs=30, steps_per_epoch=n_train_steps, validation_data=val_ds, validation_steps=n_val_steps, callbacks=[reduce_lr])